# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access basic metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list all record sets in the dataset, along with their `@id`s and the `@id`s of their fields.

In [ ]:
# List all record sets with their @id
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- Name: {getattr(rs, 'name', 'N/A')}")
    print(f"  @id: {rs.id}")
    # Show fields for each record set
    print("  Fields/Columns:")
    for field in rs.fields:
        print(f"    - Name: {getattr(field, 'name', 'N/A')}, @id: {field.id}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract **all record sets** and preview the first one as an example. This ensures all '@id's are used for referencing.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Each tuple is a dictionary (column_name: value), columns are field .id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id}, shape: {df.shape}")

# Preview first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"First record set ID: {first_rs}")
    print("Available columns (field @ids):")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Let's select a **numeric field** from the primary protein measurements record set and perform filtering, normalization, and grouping. All references use `@id` values.

In [ ]:
# Choose a record set and a numeric field by @id for demonstration
# (Change these based on printed results from above cell!)
# Example for illustration; Please update as needed by exploring actual output.
record_set_id = record_set_ids[0]  # Default to first, change if needed
df = dataframes[record_set_id]

# Try to guess a common numeric field by matching typical names
numeric_field_id = None
for field in dataset.metadata.record_sets[0].fields:
    # Look for peptide count, molecular weight, or abundance
    if any(sub in getattr(field, 'name', '').lower() for sub in ["abundance", "count", "coverage", "weight", "mw"]):
        numeric_field_id = field.id
        break
if numeric_field_id is None:
    # Else fallback to the first column
    numeric_field_id = df.columns[0]

# Check if the field is numeric, if not, try converting
if not pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean()  # Use mean as a filtering example
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id]].head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a common categorical field; look for 'description', 'accession', or similar
group_field_id = None
for field in dataset.metadata.record_sets[0].fields:
    if any(sub in getattr(field, 'name', '').lower() for sub in ["description", "accession", "class", "sample", "id"]):
        if field.id != numeric_field_id and field.id in df.columns:
            group_field_id = field.id
            break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the numeric field and, if a group field was found, show group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If group_field_id exists, plot mean per group (barplot)
if group_field_id:
    plt.figure(figsize=(10,4))
    # Show top 20 groups by mean
    top_groups = grouped_df.sort_values(by=numeric_field_id, ascending=False).head(20)
    sns.barplot(y=group_field_id, x=numeric_field_id, data=top_groups)
    plt.title(f'Mean {numeric_field_id} per {group_field_id} (Top 20)')
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset using `mlcroissant`, explored its record sets and fields via their `@id`s, extracted and analyzed data, and visualized key summary statistics. All references to dataset structure (record set, field, and column) were made using their stable `@id` values, allowing reliable programmatic access. 

You can now proceed to more specific analyses, machine learning, or additional visualization as required for your scientific or research goals.